# Metrics

> Metric tracking and analysis tools

This module provides a comprehensive suite of metric tracking and analysis tools. It is designed to evaluate model performance across diverse biomedical imaging tasks by seamlessly wrapping native MONAI metrics into fastai-compatible formats via a unified `get_metric` adapter. 

**Key Evaluation Categories:**

* **Regression Metrics:** Includes standard image quality and error evaluation tools such as Structural Similarity Index Measure (`SSIMMetric`), Peak Signal To Noise Ratio (`PSNRMetric`), Multi-Scale SSIM (`MSSSIMMetric`), Mean Absolute Error (`MAEMetric`), and Root Mean Squared Error (`RMSEMetric`).
* **Segmentation Metrics:** Features robust evaluation functions tailored for biological structures, including binary and instance segmentation tracking (`DiceMetric`), as well as panoptic map evaluation (`PanopticQualityMetric`).
* **Classification Metrics:** Provides area under the ROC curve calculations (`ROCAUCMetric`), capable of automatically handling probability activations and one-hot encoded targets.
* **Metrics Reloaded:** Integrates the advanced "Metrics Reloaded" framework (`MetricsReloadedBinary`, `MetricsReloadedCategorical`). This system helps researchers avoid traditional validation pitfalls by recommending appropriate metrics based on a task's unique "problem fingerprint" (e.g., class prevalence, boundary importance).
* **Fourier Ring Correlation (FRC):** Features a specialized frequency-domain metric (`FRCMetric`) derived from `FRCLoss`. It is widely used in super-resolution imaging and cryo-electron microscopy to estimate spatial resolution by quantifying the similarity between independent measurements.

In [ ]:
#| default_exp metrics

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

from types import SimpleNamespace
from torch import rand as torchrand, ones as torchones, randn as torchrandn
from fastai.learner import Learner as fastaiLearner


In [ ]:
#| export
# =================================
# PyTorch
# =================================
from torch import (
    abs,
    argmax,
    complex64,
    div,
    isnan,
    is_tensor,
    isinf,
    real,
    sigmoid,
    where,
    zeros_like,
)
from torch.nn.functional import one_hot, softmax

# =================================
# fastai
# =================================
import fastai.metrics as fm
from fastai.vision.all import AvgMetric, Metric, partial

# =================================
# MONAI
# =================================
import monai.metrics as mm

# =================================
# bioMONAI
# =================================
from bioMONAI.losses import FRCLoss
from bioMONAI.utils import *

## Base Classes

### Metric registry

In [ ]:
#| export

METRIC_BACKENDS = {}


def register_metric_backend(name):
    """Register a metric backend."""
    def decorator(backend):
        if name in METRIC_BACKENDS:
            raise ValueError(f"Metric backend already registered: {name}")
        METRIC_BACKENDS[name] = backend
        return backend

    return decorator

### Backend Adapters

In [ ]:
#| export
class MonaiFastaiMetric(Metric):
    """
    Adapt a MONAI metric to the fastai metric interface.

    This adapter delegates metric computation to a MONAI metric while
    providing the interface expected by fastai's training loop.

    The prediction and target are obtained from ``learn.pred`` and
    ``learn.yb[0]`` respectively. Subclasses can override ``_prepare``
    to adapt the data representation required by a specific MONAI metric.

    Parameters
    ----------
    metric : Metric
        Instantiated MONAI metric.
    name : str, optional
        Name exposed to fastai. If omitted, the ``"Metric"`` suffix is
        removed from the MONAI metric class name.

    Notes
    -----
    This class does not alter predictions or targets by default. Metric-
    specific preprocessing should be implemented by overriding
    ``_prepare`` in a subclass.
    """

    def __init__(self, metric, name=None):
        self.metric = metric
        self._name = name or metric.__class__.__name__.removesuffix("Metric")

    @property
    def name(self):
        return self._name

    def reset(self):
        self.metric.reset()

    def _prepare(self, pred, target):
        """
        Prepare predictions and targets before metric computation.

        Parameters
        ----------
        pred : torch.Tensor
            Predictions produced by the fastai learner.
        target : torch.Tensor
            Ground-truth target from the fastai learner.

        Returns
        -------
        tuple
            Prepared ``(pred, target)`` pair.

        Notes
        -----
        The default implementation returns the inputs unchanged. Subclasses
        can override this method when a MONAI metric requires a specific
        representation.
        """
        return pred, target

    def accumulate(self, learn):
        pred, target = self._prepare(
            learn.pred,
            learn.yb,
        )
        self.metric(pred, target)

    @property
    def value(self):
        value = self.metric.aggregate()

        if isinstance(value, tuple):
            value = value[0]

        if hasattr(value, "numel") and value.numel() == 1:
            return value.item()

        return value

In [ ]:
#| export 

class MetricBackend:
    """Base class for a metric backend."""

    def create(self, metric_cls, *args, **kwargs):
        return metric_cls(*args, **kwargs)

# monai

@register_metric_backend("monai")
class MonaiMetricBackend(MetricBackend):
    pass


# fastai 

@register_metric_backend("fastai")
class FastaiMetricBackend(MetricBackend):

    def create(self, metric_cls, *args, **kwargs):
        metric = metric_cls(*args, **kwargs)

        if isinstance(metric, Metric):
            return metric

        return MonaiFastaiMetric(metric)


### Base metric


In [ ]:
#| export

class BioMetric:
    default_backend = "monai"
    _default = None

    @classmethod
    def _get_metric(cls, backend):
        metric_cls = getattr(cls, f"_{backend}", None)

        if metric_cls is None:
            metric_cls = cls._default

        if metric_cls is None:
            raise ValueError(
                f"No metric implementation for backend '{backend}' "
                f"and no default implementation defined."
            )

        return metric_cls

    @classmethod
    def _create_metric(cls, backend, *args, **kwargs):
        metric_cls = cls._get_metric(backend)

        try:
            backend_cls = METRIC_BACKENDS[backend]
        except KeyError:
            raise ValueError(f"Unknown metric backend: {backend}")

        return backend_cls().create(metric_cls, *args, **kwargs)

    @property
    def name(self):
        return self.__class__.__name__

    def __new__(cls, *args, backend=None, **kwargs):
        backend = backend or cls.default_backend
        return cls._create_metric(backend, *args, **kwargs)

## Regression metrics

This section provides a robust collection of standard regression and image quality assessment metrics. By leveraging the `get_metric` adapter, these functions wrap native MONAI metrics—such as Structural Similarity Index Measure (SSIM), Peak Signal-to-Noise Ratio (PSNR), Multi-Scale SSIM (MS-SSIM), Mean Absolute Error (MAE), and Root Mean Squared Error (RMSE)—ensuring they are seamlessly compatible with multiple execution backends, including the fastai training loop.

In [ ]:
#| export

class MSEMetric(BioMetric):
    _default = mm.MSEMetric


class SSIMMetric(BioMetric):
    _default = mm.SSIMMetric


class PSNRMetric(BioMetric):
    _default = mm.PSNRMetric


class MSSSIMMetric(BioMetric):
    _default = mm.MultiScaleSSIMMetric


class MAEMetric(BioMetric):
    _default = mm.MAEMetric


class RMSEMetric(BioMetric):
    _default = mm.RMSEMetric

In [ ]:
show_doc(mm.MSEMetric)

---

### SSIMMetric

```python

def SSIMMetric(
    spatial_dims:int, data_range:float=1.0, kernel_type:KernelType | str=gaussian, win_size:int | Sequence[int]=11,
    kernel_sigma:float | Sequence[float]=1.5, k1:float=0.01, k2:float=0.03, reduction:MetricReduction | str=mean,
    get_not_nans:bool=False
)->None:


```

*Computes the Structural Similarity Index Measure (SSIM).*

.. math::
    \operatorname {SSIM}(x,y) =\frac {(2 \mu_x \mu_y + c_1)(2 \sigma_{xy} + c_2)}{((\mu_x^2 + \
            \mu_y^2 + c_1)(\sigma_x^2 + \sigma_y^2 + c_2)}

For more info, visit
    https://vicuesoft.com/glossary/term/ssim-ms-ssim/

SSIM reference paper:
    Wang, Zhou, et al. "Image quality assessment: from error visibility to structural
    similarity." IEEE transactions on image processing 13.4 (2004): 600-612.

Args:
    spatial_dims: number of spatial dimensions of the input images.
    data_range: value range of input images. (usually 1.0 or 255)
    kernel_type: type of kernel, can be "gaussian" or "uniform".
    win_size: window size of kernel
    kernel_sigma: standard deviation for Gaussian kernel.
    k1: stability constant used in the luminance denominator
    k2: stability constant used in the contrast denominator
    reduction: define the mode to reduce metrics, will only execute reduction on `not-nan` values,
        available reduction modes: {``"none"``, ``"mean"``, ``"sum"``, ``"mean_batch"``, ``"sum_batch"``,
        ``"mean_channel"``, ``"sum_channel"``}, default to ``"mean"``. if "none", will not do reduction
    get_not_nans: whether to return the `not_nans` count, if True, aggregate() returns (metric, not_nans)

In [ ]:
show_doc(mm.SSIMMetric)

---

### SSIMMetric

```python

def SSIMMetric(
    spatial_dims:int, data_range:float=1.0, kernel_type:KernelType | str=gaussian, win_size:int | Sequence[int]=11,
    kernel_sigma:float | Sequence[float]=1.5, k1:float=0.01, k2:float=0.03, reduction:MetricReduction | str=mean,
    get_not_nans:bool=False
)->None:


```

*Computes the Structural Similarity Index Measure (SSIM).*

.. math::
    \operatorname {SSIM}(x,y) =\frac {(2 \mu_x \mu_y + c_1)(2 \sigma_{xy} + c_2)}{((\mu_x^2 + \
            \mu_y^2 + c_1)(\sigma_x^2 + \sigma_y^2 + c_2)}

For more info, visit
    https://vicuesoft.com/glossary/term/ssim-ms-ssim/

SSIM reference paper:
    Wang, Zhou, et al. "Image quality assessment: from error visibility to structural
    similarity." IEEE transactions on image processing 13.4 (2004): 600-612.

Args:
    spatial_dims: number of spatial dimensions of the input images.
    data_range: value range of input images. (usually 1.0 or 255)
    kernel_type: type of kernel, can be "gaussian" or "uniform".
    win_size: window size of kernel
    kernel_sigma: standard deviation for Gaussian kernel.
    k1: stability constant used in the luminance denominator
    k2: stability constant used in the contrast denominator
    reduction: define the mode to reduce metrics, will only execute reduction on `not-nan` values,
        available reduction modes: {``"none"``, ``"mean"``, ``"sum"``, ``"mean_batch"``, ``"sum_batch"``,
        ``"mean_channel"``, ``"sum_channel"``}, default to ``"mean"``. if "none", will not do reduction
    get_not_nans: whether to return the `not_nans` count, if True, aggregate() returns (metric, not_nans)

In [ ]:
show_doc(mm.PSNRMetric)

---

### PSNRMetric

```python

def PSNRMetric(
    max_val:int | float, reduction:MetricReduction | str=mean, get_not_nans:bool=False
)->None:


```

*Compute Peak Signal To Noise Ratio between two tensors using function:*

.. math::
    \operatorname{PSNR}\left(Y, \hat{Y}\right) = 20 \cdot \log_{10} \left({\mathit{MAX}}_Y\right) \
    -10 \cdot \log_{10}\left(\operatorname{MSE\left(Y, \hat{Y}\right)}\right)

More info: https://en.wikipedia.org/wiki/Peak_signal-to-noise_ratio

Help taken from:
https://github.com/tensorflow/tensorflow/blob/master/tensorflow/python/ops/image_ops_impl.py line 4139

Input `y_pred` is compared with ground truth `y`.
Both `y_pred` and `y` are expected to be real-valued, where `y_pred` is output from a regression model.

Example of the typical execution steps of this metric class follows :py:class:`monai.metrics.metric.Cumulative`.

Args:
    max_val: The dynamic range of the images/volumes (i.e., the difference between the
        maximum and the minimum allowed values e.g. 255 for a uint8 image).
    reduction: define the mode to reduce metrics, will only execute reduction on `not-nan` values,
        available reduction modes: {``"none"``, ``"mean"``, ``"sum"``, ``"mean_batch"``, ``"sum_batch"``,
        ``"mean_channel"``, ``"sum_channel"``}, default to ``"mean"``. if "none", will not do reduction.
    get_not_nans: whether to return the `not_nans` count, if True, aggregate() returns (metric, not_nans).

In [ ]:
show_doc(mm.MAEMetric)

---

### MAEMetric

```python

def MAEMetric(
    reduction:MetricReduction | str=mean, get_not_nans:bool=False
)->None:


```

*Compute Mean Absolute Error between two tensors using function:*

.. math::
    \operatorname {MAE}\left(Y, \hat{Y}\right) =\frac {1}{n}\sum _{i=1}^{n}\left|y_i-\hat{y_i}\right|.

More info: https://en.wikipedia.org/wiki/Mean_absolute_error

Input `y_pred` is compared with ground truth `y`.
Both `y_pred` and `y` are expected to be real-valued, where `y_pred` is output from a regression model.

Example of the typical execution steps of this metric class follows :py:class:`monai.metrics.metric.Cumulative`.

Args:
    reduction: define the mode to reduce metrics, will only execute reduction on `not-nan` values,
        available reduction modes: {``"none"``, ``"mean"``, ``"sum"``, ``"mean_batch"``, ``"sum_batch"``,
        ``"mean_channel"``, ``"sum_channel"``}, default to ``"mean"``. if "none", will not do reduction.
    get_not_nans: whether to return the `not_nans` count, if True, aggregate() returns (metric, not_nans).

In [ ]:
show_doc(mm.RMSEMetric)

---

### RMSEMetric

```python

def RMSEMetric(
    reduction:MetricReduction | str=mean, get_not_nans:bool=False
)->None:


```

*Compute Root Mean Squared Error between two tensors using function:*

.. math::
    \operatorname {RMSE}\left(Y, \hat{Y}\right) ={ \sqrt{ \frac {1}{n}\sum _{i=1}^{n}\left(y_i-\hat{y_i}\right)^2 } } \
    = \sqrt {\operatorname{MSE}\left(Y, \hat{Y}\right)}.

More info: https://en.wikipedia.org/wiki/Root-mean-square_deviation

Input `y_pred` is compared with ground truth `y`.
Both `y_pred` and `y` are expected to be real-valued, where `y_pred` is output from a regression model.

Example of the typical execution steps of this metric class follows :py:class:`monai.metrics.metric.Cumulative`.

Args:
    reduction: define the mode to reduce metrics, will only execute reduction on `not-nan` values,
        available reduction modes: {``"none"``, ``"mean"``, ``"sum"``, ``"mean_batch"``, ``"sum_batch"``,
        ``"mean_channel"``, ``"sum_channel"``}, default to ``"mean"``. if "none", will not do reduction.
    get_not_nans: whether to return the `not_nans` count, if True, aggregate() returns (metric, not_nans).

---

#### MSSIMMetric

```python

def MSSIMMetric(
    spatial_dims:int, data_range:float=1.0, kernel_type:KernelType | str=gaussian,
    kernel_size:int | Sequence[int]=11, kernel_sigma:float | Sequence[float]=1.5, k1:float=0.01, k2:float=0.03,
    weights:Sequence[float]=(0.0448, 0.2856, 0.3001, 0.2363, 0.1333), reduction:MetricReduction | str=mean,
    get_not_nans:bool=False
)->None:


```

*Computes the Multi-Scale Structural Similarity Index Measure (MS-SSIM).*

MS-SSIM reference paper:
    Wang, Z., Simoncelli, E.P. and Bovik, A.C., 2003, November. "Multiscale structural
    similarity for image quality assessment." In The Thirty-Seventh Asilomar Conference
    on Signals, Systems & Computers, 2003 (Vol. 2, pp. 1398-1402). IEEE

Args:
    spatial_dims: number of spatial dimensions of the input images.
    data_range: value range of input images. (usually 1.0 or 255)
    kernel_type: type of kernel, can be "gaussian" or "uniform".
    kernel_size: size of kernel
    kernel_sigma: standard deviation for Gaussian kernel.
    k1: stability constant used in the luminance denominator
    k2: stability constant used in the contrast denominator
    weights: parameters for image similarity and contrast sensitivity at different resolution scores.
    reduction: define the mode to reduce metrics, will only execute reduction on `not-nan` values,
        available reduction modes: {``"none"``, ``"mean"``, ``"sum"``, ``"mean_batch"``, ``"sum_batch"``,
        ``"mean_channel"``, ``"sum_channel"``}, default to ``"mean"``. if "none", will not do reduction
    get_not_nans: whether to return the `not_nans` count, if True, aggregate() returns (metric, not_nans)

#### Example: Using regression metrics with different backends


In [ ]:
# 1. Instantiate metrics using the default MONAI backend
mae_native = MAEMetric(backend="monai", reduction="mean")
ssim_native = SSIMMetric(backend="monai", spatial_dims=2, data_range=1.0)

# 2. Instantiate a metric wrapped for the fastai training loop
mae_fastai = MAEMetric(backend="fastai")

# --- Native Backend Usage ---
# Create dummy prediction and target tensors
preds = torchTensor([[1.0, 2.5], [3.0, 4.0]])
targs = torchTensor([[1.0, 2.0], [3.0, 4.0]])

mae_native.reset()
mae_native(preds, targs)
print(f"Native MAE Score: {mae_native.aggregate().item()}") # Expected: 0.125 (only 0.5 diff on one element)

# --- Fastai Backend Usage ---
# Fastai uses a `learn` object internally, which we mock here
mae_fastai.reset()
mae_fastai.accumulate(TestLearner(pred=preds, yb=targs))
print(f"Fastai MAE Score: {mae_fastai.value}")


Native MAE Score: 0.125
Fastai MAE Score: 0.125


In [ ]:
#| hide

def test_regression_metrics_partials():
    # --- Test 1: MAEMetric (Simple distance metric) ---
    pred1 = torchTensor([[2.0, 3.0]])
    targ1 = torchTensor([[2.0, 2.0]]) # Difference of 1.0 on one element, mean = 0.5
    
    # Native
    mae_monai = MAEMetric(backend="monai")
    mae_monai(pred1, targ1)
    test_close(mae_monai.aggregate().item(), 0.5, eps=1e-5)
    
    # Fastai wrapper
    mae_fastai = MAEMetric(backend="fastai")
    test_eq(mae_fastai.name, "MAE")
    
    mae_fastai.accumulate(TestLearner(pred=pred1, yb=targ1))
    test_close(mae_fastai.value, 0.5, eps=1e-5)

    # --- Test 2: SSIMMetric (Complex metric requiring spatial_dims) ---
    # SSIM requires a 4D tensor (B, C, H, W) for 2D spatial dims
    pred2 = torchones(1, 1, 16, 16)
    targ2 = torchones(1, 1, 16, 16) # Exact match -> SSIM = 1.0
    
    ssim_monai = SSIMMetric(backend="monai", spatial_dims=2, data_range=1.0)
    ssim_monai(pred2, targ2)
    test_close(ssim_monai.aggregate().item(), 1.0, eps=1e-4)

    return "All regression partial tests passed"

test_eq(test_regression_metrics_partials(), "All regression partial tests passed")

#### `SSIMMetric` Parameter Reference

*Computes the Structural Similarity Index Measure (SSIM).*

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`spatial_dims`** | `int` | *Required* | Number of spatial dimensions of the input images. |
| **`data_range`** | `float` | `1.0` | Value range of input images (usually `1.0` or `255.0`). |
| **`kernel_type`** | `str` | `"gaussian"` | Type of kernel, can be `"gaussian"` or `"uniform"`. |
| **`win_size`** | `int \| list` | `11` | Window size of the kernel. |
| **`kernel_sigma`** | `float \| list` | `1.5` | Standard deviation for the Gaussian kernel. |
| **`k1`** | `float` | `0.01` | Stability constant used in the luminance denominator. |
| **`k2`** | `float` | `0.03` | Stability constant used in the contrast denominator. |
| **`reduction`** | `str` | `"mean"` | Reduction mode (e.g., `"mean"`, `"sum"`, `"none"`). Executed only on non-NaN values. |
| **`get_not_nans`** | `bool` | `False` | Whether to return the `not_nans` count alongside the metric. |
| **`backend`** | `str` | `"monai"` | *bioMONAI specific:* Target execution backend (`"monai"`, `"ignite"`, or `"fastai"`). |

---

#### `PSNRMetric` Parameter Reference

*Computes the Peak Signal To Noise Ratio (PSNR).*

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`max_val`** | `int \| float` | *Required* | The dynamic range of the images (the difference between the max and min allowed values, e.g., `255`). |
| **`reduction`** | `str` | `"mean"` | Reduction mode (e.g., `"mean"`, `"sum"`, `"none"`). |
| **`get_not_nans`** | `bool` | `False` | Whether to return the `not_nans` count alongside the metric. |
| **`backend`** | `str` | `"monai"` | *bioMONAI specific:* Target execution backend (`"monai"`, `"ignite"`, or `"fastai"`). |

---

#### `MSSSIMMetric` Parameter Reference

*Computes the Multi-Scale Structural Similarity Index Measure (MS-SSIM).*

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`spatial_dims`** | `int` | *Required* | Number of spatial dimensions of the input images. |
| **`data_range`** | `float` | `1.0` | Value range of input images. |
| **`kernel_type`** | `str` | `"gaussian"` | Type of kernel, can be `"gaussian"` or `"uniform"`. |
| **`kernel_size`** | `int \| list` | `11` | Size of the kernel. |
| **`kernel_sigma`** | `float \| list` | `1.5` | Standard deviation for the Gaussian kernel. |
| **`k1`** | `float` | `0.01` | Stability constant used in the luminance denominator. |
| **`k2`** | `float` | `0.03` | Stability constant used in the contrast denominator. |
| **`weights`** | `list` | `(0.04..., ...)`| Parameters for image similarity and contrast sensitivity at different resolutions. |
| **`reduction`** | `str` | `"mean"` | Reduction mode (e.g., `"mean"`, `"sum"`, `"none"`). |
| **`get_not_nans`** | `bool` | `False` | Whether to return the `not_nans` count alongside the metric. |
| **`backend`** | `str` | `"monai"` | *bioMONAI specific:* Target execution backend (`"monai"`, `"ignite"`, or `"fastai"`). |

---

#### `MAEMetric` Parameter Reference

*Computes the Mean Absolute Error (MAE).*

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`reduction`** | `str` | `"mean"` | Reduction mode (e.g., `"mean"`, `"sum"`, `"none"`). |
| **`get_not_nans`** | `bool` | `False` | Whether to return the `not_nans` count alongside the metric. |
| **`backend`** | `str` | `"monai"` | *bioMONAI specific:* Target execution backend (`"monai"`, `"ignite"`, or `"fastai"`). |

---

#### `RMSEMetric` Parameter Reference

*Computes the Root Mean Squared Error (RMSE).*

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`reduction`** | `str` | `"mean"` | Reduction mode (e.g., `"mean"`, `"sum"`, `"none"`). |
| **`get_not_nans`** | `bool` | `False` | Whether to return the `not_nans` count alongside the metric. |
| **`backend`** | `str` | `"monai"` | *bioMONAI specific:* Target execution backend (`"monai"`, `"ignite"`, or `"fastai"`). |

## Segmentation metrics

This section provides wrapper functions for evaluating segmentation tasks. These metrics automatically handle common tensor shape adjustments, activation functions (like sigmoid), and binarization steps before passing the data to the underlying evaluation engines.

**Available Metrics:**

* **`DiceMetric`**: Dice coefficient metric for binary segmentation with 1-channel logits.
* **`PanopticQualityMetric`**: evaluates panoptic maps containing encoded semantic and instance IDs.

#### `DiceMetric` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`threshold`** | `float` | `0.5` | Threshold applied to model predictions to generate binary masks. (If `instance=False`, logits are passed through a sigmoid first). |
| **`instance`** | `bool` | `False` | If `True`, treats the target as an instance mask, converting all non-zero values to `1` (foreground) and binarizing predictions directly based on the threshold. |
| **`**kwargs`** | `dict` | `{}` | Additional keyword arguments forwarded to `monai.metrics.DiceMetric` (e.g., `include_background`, `reduction`, `ignore_empty`). |

In [ ]:
#| export

class DiceFastaiMetric(MonaiFastaiMetric):
    """
    Adapt MONAI's ``DiceMetric`` to the fastai metric interface.

    This adapter creates a MONAI ``DiceMetric`` and converts predictions
    produced by a fastai segmentation model to the representation expected
    by MONAI before computing the metric.

    Parameters
    ----------
    include_background : bool, default=True
        Whether to include the background class in the Dice computation.

    reduction : str, default="mean"
        Reduction method used to combine Dice scores.

    get_not_nans : bool, default=False
        Whether to return the number of non-NaN values used in the
        reduction.

    ignore_empty : bool, default=True
        Whether to ignore samples with an empty target segmentation.

    num_classes : int, optional
        Number of classes when using single-channel label maps.

    return_with_label : bool or list of str, default=False
        Whether to return Dice values together with their labels.

    per_component : bool, default=False
        Whether to compute Dice independently for each binary component.

    Notes
    -----
    For single-channel predictions, logits are converted to binary
    predictions using a sigmoid activation and a threshold of 0.5.

    For multi-channel predictions, the class with the highest logit is
    selected using ``argmax``.

    Targets without a channel dimension are expanded to include one.
    The actual Dice computation is delegated to MONAI's ``DiceMetric``.
    """

    def __init__(self, *args, **kwargs):
        super().__init__(
            mm.DiceMetric(*args, **kwargs),
            name="Dice",
        )

    def _prepare(self, pred, target):
        # Binary segmentation: one output channel
        if pred.ndim == target.ndim + 1:
            if pred.shape[1] == 1:
                pred = (torch.sigmoid(pred) > 0.5).float()

            # Multi-class segmentation
            else:
                pred = pred.argmax(dim=1, keepdim=True)

        # Add channel dimension to label-map targets
        if target.ndim == pred.ndim - 1:
            target = target.unsqueeze(1)

        return pred, target

In [ ]:
#| export

class DiceMetric(BioMetric):
    """
    Compute the Dice coefficient for segmentation tasks.

    The metric measures the overlap between predicted and target
    segmentations. It supports both single-channel label maps for
    multi-class segmentation and multi-channel predictions for
    multi-label or class-wise segmentation.

    Parameters
    ----------
    include_background : bool, default=True
        Whether to include the background channel in the computation.

    reduction : str, default="mean"
        Reduction method applied to the per-sample and per-channel
        Dice scores. Common options include "mean", "sum",
        "mean_batch", "sum_batch", and "none".

    get_not_nans : bool, default=False
        If True, also return the number of non-NaN values used in
        the reduction.

    ignore_empty : bool, default=True
        Whether to ignore cases where the target segmentation is empty.
        If True, empty target regions do not contribute to the metric.

    num_classes : int, optional
        Number of classes when the input is provided as a single-channel
        label map. This is required when the number of classes cannot be
        inferred from the input.

    return_with_label : bool or list of str, default=False
        If True, return the metric value together with its class labels.
        A list of labels can be provided to explicitly name the classes.

    per_component : bool, default=False
        If True, compute the Dice score independently for each binary
        component.

    Examples
    --------
    Compute the mean Dice score while excluding the background::

        dice = DiceMetric(
            include_background=False,
            reduction="mean",
        )

    Compute per-class Dice scores::

        dice = DiceMetric(
            include_background=False,
            reduction="none",
        )
    """
    _default = mm.DiceMetric
    _fastai = DiceFastaiMetric



#### Example: Using DiceMetric 

In [ ]:
# 1. Instantiate the Metric wrapper
# include_background=True ensures we calculate the score for the whole mask in this simple test
dice = DiceMetric(backend="monai")

# 2. Simulate raw output logits and target masks
# Logits > 0 will result in sigmoid probabilities > 0.5
pred_logits = torchTensor([[[[10.0, -10.0], [10.0, -10.0]]]]) # Represents foreground left, background right
pred_mask = (sigmoid(pred_logits)>.5).float()
target_mask = torchTensor([[[[1.0, 0.0], [1.0, 1.0]]]])       # Target differs in bottom-right pixel

# 3. Fastai accumulates metrics over batches via a 'learn' object.
# We mock it completely with all the attributes AvgMetric expects (pred, y, yb, and to_detach)
dice_value = float(dice(pred_mask, target_mask))

print(f"Calculated Dice Score:   {dice_value}")
# Expected output:
# Calculated Dice Score:   0.800000011920929

Calculated Dice Score:   0.800000011920929


In [ ]:
pred_logits = torchTensor([
    [
        [[-10.,  10.],
         [-10.,  10.]],   # background

        [[ 10., -10.],
         [ 10., -10.]],   # foreground
    ]
])

target = torchTensor([
    [[1., 0.],
     [1., 1.]]
])

dice = DiceMetric(backend="fastai", include_background=False)

dice.reset()

learn = TestLearner(pred=pred_logits, yb=target)

dice.accumulate(learn)

# print("inter:", dice.inter)
# print("union:", dice.union)
print(f"Calculated Dice Score:   {dice.value}")



Calculated Dice Score:   0.800000011920929


In [ ]:
#| hide

test_close(dice_value,0.8)
test_close(dice.value,0.8)

In [ ]:
#| export
class IoUMetric(BioMetric):
    _default = mm.MeanIoU

In [ ]:
#| export
class GeneralizedDiceScore(BioMetric):
    _default = mm.GeneralizedDiceScore

#### `PanopticQualityMetric` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`**kwargs`** | `dict` | `{}` | All keyword arguments are passed directly to the underlying `monai.metrics.PanopticQualityMetric` (e.g., `num_classes`, `match_iou_threshold`). |

In [ ]:
#| export

class PanopticQualityMetric(BioMetric):
    """
    Wrapper around monai.metrics.PanopticQualityMetric.

    Expects:
        pred  : (B, C, H, W) logits
        target: (B, H, W) or (B, 1, H, W) panoptic map
                Each pixel contains encoded panoptic label
                (semantic + instance id).

    All kwargs are forwarded to MONAI PanopticQualityMetric.
    """
    _default = mm.PanopticQualityMetric
    # pq_metric = mm.PanopticQualityMetric(**kwargs)

    # def PQ(pred, target):
    #     # Convert logits to discrete labels
    #     # pred = pred.argmax(dim=1)  # (B, H, W)

    #     if target.ndim == 4 and target.shape[1] == 1:
    #         target = target.squeeze(1)

    #     pq_metric.reset()
    #     pq_metric(y_pred=pred, y=target)
    #     return pq_metric.aggregate()

    # return AvgMetric(PQ)

#### Example: Using PanopticQualityMetric in a fastai workflow

In [ ]:
# 1. Instantiate the Metric wrapper
# MONAI's PanopticQualityMetric strictly requires the 'num_classes' argument
pq_metric = PanopticQualityMetric(backend="fastai", num_classes=3)

# 2. Simulate raw output and target panoptic maps
pred_panoptic = torchTensor([
    [
        [[1, 1], [2, 0]],  # semantic labels
        [[1, 1], [1, 0]],  # instance IDs
    ]
])

target_panoptic = torchTensor([
    [
        [[1, 1], [2, 0]],  # semantic labels
        [[1, 1], [1, 0]],  # instance IDs
    ]
])

# 3. Mock the fastai learner object correctly
mock_learn_pq = TestLearner(preds=pred_panoptic, targs=target_panoptic)

pq_metric.reset()
pq_metric.accumulate(mock_learn_pq)

print(f"Panoptic Quality Name:  {pq_metric.name}")
print(f"Panoptic Quality Score: {pq_metric.value}")


Panoptic Quality Name:  PanopticQuality
Panoptic Quality Score: tensor([1.0000, 0.0000, 0.0000])


In [ ]:
#| hide
val = pq_metric.value
test_eq(val is not None, True)

## Classification Metrics

This section provides metrics tailored for classification tasks. These wrappers are designed to smoothly integrate into the fastai ecosystem, handling common classification preprocessing steps such as applying activation functions and generating one-hot encodings automatically.

**Available Metrics:**

* **`ROCAUCMetric`**: Computes the Area Under the Receiver Operating Characteristic Curve (ROC AUC), with built-in support for probability activations (e.g., Sigmoid, Softmax) and automatic one-hot encoding for target labels.

In [ ]:
#| export

def ROCAUCMetric(num_classes=None, # if not None, checks if preds and targets are one-hot encoded
                 act=None,         # activation operations, typically Sigmoid or Softmax.
                 **kwargs):
    """
    Wrapper around monai.metrics.ROCAUCMetric.

    If num_classes is None:
        assumes pred and target are already one-hot / probability encoded.

    If num_classes is provided:
        automatically one-hot encodes pred/target when needed.
    """
    rocaucmetric = mm.ROCAUCMetric(**kwargs)

    def _maybe_one_hot(x):
        # already encoded
        if x.ndim > 1 and x.shape[1] == num_classes:
            return x

        return one_hot(x.long(), num_classes=num_classes)

    def ROCAUC(pred, target):
        if act is not None:
            pred = act(pred)

        if num_classes is not None:
            pred = _maybe_one_hot(pred)
            target = _maybe_one_hot(target)

        rocaucmetric.reset()
        rocaucmetric(pred, target)
        return rocaucmetric.aggregate()

    return AvgMetric(ROCAUC)

In [ ]:
#| export

class ROCAUCFastaiMetric(MonaiFastaiMetric):
    """
    Adapt MONAI's ``ROCAUCMetric`` to the fastai metric interface.

    This adapter optionally applies an activation function and converts
    class-index predictions and targets to one-hot representations before
    delegating ROC-AUC computation to MONAI.

    Parameters
    ----------
    num_classes : int, optional
        Number of classes.

        If provided, predictions and targets are converted to one-hot
        representations when they are not already encoded with
        ``num_classes`` channels.

        If ``None``, predictions and targets are assumed to already have
        the representation expected by MONAI's ``ROCAUCMetric``.

    act : callable, optional
        Activation function applied to predictions before computing the
        metric. Typical choices are ``torch.sigmoid`` for binary
        classification or ``torch.softmax`` for multiclass
        classification.

    *args
        Positional arguments passed to ``monai.metrics.ROCAUCMetric``.

    **kwargs
        Additional keyword arguments passed to
        ``monai.metrics.ROCAUCMetric``.

    Notes
    -----
    The adapter detects already encoded data by checking whether the
    second dimension contains ``num_classes`` channels. Otherwise,
    class-index labels are converted to one-hot representations.

    The underlying MONAI metric is reset before each accumulation step.
    This preserves the behavior of the previous function-based wrapper,
    where each fastai metric evaluation produced an independent ROC-AUC
    value.

    Examples
    --------
    Binary classification with sigmoid activation::

        metric = ROCAUCFastaiMetric(
            num_classes=2,
            act=torch.sigmoid,
        )

    Multiclass classification with softmax activation::

        metric = ROCAUCFastaiMetric(
            num_classes=3,
            act=lambda x: torch.softmax(x, dim=1),
        )
    """

    def __init__(self, num_classes=None, act=None, *args, **kwargs):
        self.num_classes = num_classes
        self.act = act

        metric = mm.ROCAUCMetric(*args, **kwargs)

        super().__init__(
            metric,
            name="ROCAUC",
        )

    def _maybe_one_hot(self, x):
        """
        Convert class-index data to one-hot representation when needed.

        Parameters
        ----------
        x : torch.Tensor
            Predictions or targets.

        Returns
        -------
        torch.Tensor
            ``x`` unchanged if it already contains ``num_classes``
            channels; otherwise converted to one-hot representation.
        """
        if self.num_classes is None:
            return x

        if x.ndim > 1 and x.shape[1] == self.num_classes:
            return x

        return one_hot(
            x.long(),
            num_classes=self.num_classes,
        )

    def _prepare(self, pred, target):
        """
        Apply activation and optional one-hot encoding.

        Parameters
        ----------
        pred : torch.Tensor
            Predictions produced by the fastai learner.

        target : torch.Tensor or tuple
            Ground-truth targets. Fastai normally stores targets in
            ``learn.yb`` as a tuple.

        Returns
        -------
        tuple
            Prepared ``(pred, target)`` pair suitable for MONAI's
            ``ROCAUCMetric``.
        """
        if isinstance(target, (tuple, list)):
            target = target[0]

        if self.act is not None:
            pred = self.act(pred)

        if self.num_classes is not None:
            pred = self._maybe_one_hot(pred)
            target = self._maybe_one_hot(target)

        return pred, target

    def accumulate(self, learn):
        """
        Accumulate ROC-AUC for the current fastai batch.

        The MONAI metric is reset before processing the batch so that
        the result matches the behavior of the previous functional
        ``ROCAUCMetric`` wrapper.
        """
        pred, target = self._prepare(
            learn.pred,
            learn.yb,
        )

        self.metric.reset()
        self.metric(pred, target)

#### `ROCAUCMetric` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`num_classes`** | `int` \| `None` | `None` | If provided, automatically one-hot encodes the predictions and targets to this number of classes. If `None`, assumes the inputs are already one-hot encoded or probability arrays. |
| **`act`** | `callable` \| `None` | `None` | An activation function applied to the predictions before metric calculation (e.g., `torch.sigmoid`, `torch.nn.functional.softmax`). |
| **`**kwargs`** | `dict` | `{}` | Additional arguments forwarded directly to the underlying `monai.metrics.ROCAUCMetric` (e.g., `average`, `multi_class_mode`). |

In [ ]:
#| export

class ROCAUCMetric(BioMetric):
    """
    Compute the area under the receiver operating characteristic curve
    (ROC-AUC) using MONAI.

    ROC-AUC summarizes the ability of a model to distinguish between
    positive and negative classes across all possible classification
    thresholds. For multiclass problems, MONAI supports different
    reduction strategies for combining the per-class AUC values.

    Parameters
    ----------
    average : str, default="macro"
        Reduction method used to combine AUC values across classes.
        The available options depend on the underlying MONAI
        implementation.

    get_not_nans : bool, default=False
        If ``True``, also return the number of valid, non-NaN AUC
        values when aggregating the metric.

    kwargs
        Additional keyword arguments are passed directly to MONAI's
        :class:`monai.metrics.ROCAUCMetric`.

    Notes
    -----
    With ``backend="fastai"``, predictions are optionally activated and
    converted to one-hot representations by ``ROCAUCFastaiMetric`` before
    being passed to MONAI.

    With the default MONAI backend, ``num_classes`` and ``act`` are not
    interpreted by this wrapper; they are specific to the fastai adapter.

    Examples
    --------
    Compute macro-averaged ROC-AUC with the default MONAI backend::

        metric = ROCAUCMetric()

    Use the metric with the fastai backend::

        metric = ROCAUCMetric(backend="fastai")

    """
    _default = mm.ROCAUCMetric
    _fastai = ROCAUCFastaiMetric

#### Example: Using ROCAUCMetric in a fastai workflow

In [ ]:
# 1. Instantiate the Metric wrapper
# We specify 3 classes and apply a softmax activation on the raw logits
roc_auc = ROCAUCMetric(backend="fastai", num_classes=3, act=lambda x: softmax(x, dim=1))

# 2. Simulate raw output logits and target labels (class indices)
# We need at least one example for EACH class (0, 1, and 2) to compute AUC without warnings
pred_logits = torchTensor([
    [2.0, 0.1, 0.1],  # Predicts class 0
    [0.1, 2.5, 0.2],  # Predicts class 1
    [0.2, 0.1, 3.0]   # Predicts class 2
]) 
target_labels = torchTensor([0, 1, 2]) # All 3 classes are represented

# 3. Mock the fastai learner object correctly
mock_learn = TestLearner(
    pred=pred_logits, 
    yb=target_labels,
)

roc_auc.reset()
roc_auc.accumulate(mock_learn)

print(f"Metric Name: {roc_auc.name}")
print(f"Calculated ROC AUC: {roc_auc.value.item()}")

Metric Name: ROCAUC
Calculated ROC AUC: 1.0


In [ ]:
#| hide

def test_rocaucmetric_logic():
    # --- Test 1: With num_classes and activation (auto one-hot) ---
    roc_auc = ROCAUCMetric(backend="fastai", num_classes=2, act=sigmoid)
    
    pred1 = torchTensor([[10.0, -10.0], [-10.0, 10.0]]) # Very confident predictions
    targ1 = torchTensor([0, 1])                         # Integer class labels
    
    learn1 = SimpleNamespace(pred=pred1, y=targ1, yb=(targ1,), to_detach=lambda b: b)
    
    roc_auc.reset()
    roc_auc.accumulate(learn1)
    test_close(roc_auc.value.item(), 1.0, eps=1e-4)
    
    # --- Test 2: Pre-encoded (num_classes=None) ---
    roc_auc_pre = ROCAUCMetric(backend="fastai")
    
    pred2 = torchTensor([[0.9, 0.1], [0.1, 0.9]])       # Already probabilities
    targ2 = torchTensor([[1, 0], [0, 1]])               # Already one-hot encoded
    
    learn2 = SimpleNamespace(pred=pred2, y=targ2, yb=(targ2,), to_detach=lambda b: b)
    
    roc_auc_pre.reset()
    roc_auc_pre.accumulate(learn2)
    test_close(roc_auc_pre.value.item(), 1.0, eps=1e-4)

    return "ROCAUCMetric logic tests passed"

test_eq(test_rocaucmetric_logic(), "ROCAUCMetric logic tests passed")

In [ ]:
#| export
class AveragePrecisionMetric(BioMetric):
    _default = mm.AveragePrecisionMetric

In [ ]:
#| export
class ConfusionMatrixMetric(BioMetric):
    _default = mm.ConfusionMatrixMetric

In [ ]:
#| export
class HausdorffDistanceMetric(BioMetric):
    _default = mm.HausdorffDistanceMetric

In [ ]:
#| export
class SurfaceDistanceMetric(BioMetric):
    _default = mm.SurfaceDistanceMetric

In [ ]:
#| export
class SurfaceDiceMetric(BioMetric):
    _default = mm.SurfaceDiceMetric

## Distributions

In [ ]:
#| export
class FIDMetric(BioMetric):
    _default = mm.FIDMetric

In [ ]:
#| export
class MMDMetric(BioMetric):
    _default = mm.MMDMetric

## Metrics Reloaded

**Metrics Reloaded** is a comprehensive recommendation framework designed to help researchers and practitioners in biomedical image analysis select and apply the most appropriate performance metrics for their specific tasks. Traditional validation practices often rely on a few standard metrics (e.g., Dice score, IoU), which might not always reflect the domain-specific interests or characteristics of a particular problem — such as class imbalance, object size, boundary importance, or task type (classification, segmentation, detection). ([Nature][1])

At its core, Metrics Reloaded introduces the concept of a problem fingerprint: a structured representation of properties relevant to metric selection (e.g., whether the problem is semantic segmentation vs. object detection, the importance of boundary accuracy, class prevalence, etc.). Using the problem fingerprint, the framework **guides users through a systematic decision process** to identify a set of suitable metrics that align with both the task and domain interest. ([Nature][1])

Metrics Reloaded supports a broad range of image analysis tasks, including:

* **Image-level classification**
* **Semantic segmentation**
* **Object detection**
* **Instance segmentation**

To make the selection process more accessible, Metrics Reloaded is also available as an **interactive online tool**, where users can explore the framework’s recommendations and walk through the metric selection process based on their problem fingerprint:
[https://metrics-reloaded.dkfz.de/](https://metrics-reloaded.dkfz.de/) — *Metrics Reloaded online tool* ([metrics-reloaded.dkfz.de][2])

This tool provides a user-centric way to explore metric strengths, weaknesses, and recommendations tailored to different imaging tasks and validation challenges.

[1]: https://www.nature.com/articles/s41592-023-02151-z?utm_source=chatgpt.com "Metrics reloaded: recommendations for image analysis validation | Nature Methods"
[2]: https://metrics-reloaded.dkfz.de/?utm_source=chatgpt.com "Metrics Reloaded"


In [ ]:
#| export

class MetricsReloadedBinaryFastai(MonaiFastaiMetric):
    """
    Adapt MONAI's MetricsReloadedBinary metric to the fastai interface.

    Parameters
    ----------
    metric_name : str
        Name of the metric to compute.

    **kwargs
        Additional arguments passed to ``monai.metrics.MetricsReloadedBinary``.
    """

    def __init__(self, *args, **kwargs):
        metric = mm.MetricsReloadedBinary(*args, **kwargs)
        super().__init__(metric, name="MetricsReloadedBinary")

    def _prepare(self, pred, target):
        return pred, target[0]

In [ ]:
#| export
class MetricsReloadedBinary(BioMetric):
    """
    Compute binary segmentation metrics using the MetricsReloaded framework.

    This metric evaluates predictions for binary segmentation tasks using
    the MetricsReloaded binary evaluation framework.

    Parameters
    ----------
    spacing : tuple, optional
        Physical spacing of the image dimensions. Used when computing
        metrics that depend on physical distances.

    """

    _default = mm.MetricsReloadedBinary
    _fastai = MetricsReloadedBinaryFastai


class MetricsReloadedCategorical(BioMetric):
    """
    Compute categorical segmentation metrics using the MetricsReloaded framework.

    This metric evaluates predictions for categorical or multi-class
    segmentation tasks using the MetricsReloaded categorical evaluation
    framework.

    Parameters
    ----------
    spacing : tuple, optional
        Physical spacing of the image dimensions. Used when computing
        metrics that depend on physical distances.

    """

    _default = mm.MetricsReloadedCategorical

#### Example: Using MetricsReloadedBinary in a fastai workflow

In [ ]:
# 1. Instantiate the Metric wrapper for binary classification
mr_binary = MetricsReloadedBinary(metric_name="Accuracy", backend="fastai")

# 2. Simulate binary predictions and targets
# MONAI strictly expects at least 3 dimensions: (Batch, Channel, Spatial)
# Here we simulate a Batch of 4, 1 Channel, and 1 Spatial dimension (4, 1, 1)
pred_bin = torchTensor([[[0.9]], [[0.1]], [[0.8]], [[0.2]]])
targ_bin = torchTensor([[[1]], [[0]], [[1]], [[0]]])

# 3. Mock the fastai learner object correctly for AvgMetric
mock_learn_bin = SimpleNamespace(
    pred=pred_bin, 
    y=targ_bin,
    yb=(targ_bin,),
    to_detach=lambda b: b
)

try:
    mr_binary.reset()
    mr_binary.accumulate(mock_learn_bin)

    print(f"Metrics Reloaded Binary Name: {mr_binary.name}")
    print(f"Calculated Score: {mr_binary.value}")
except Exception as e:
    # Capturing exception in case the optional 'metricsreloaded' package is not installed
    print(f"Metrics Reloaded Exception caught: {e}")

Metrics Reloaded Exception caught: from MetricsReloaded.metrics.pairwise_measures import BinaryPairwiseMeasures (No module named 'MetricsReloaded').

For details about installing the optional dependencies, please visit:
    https://docs.monai.io/en/latest/installation.html#installing-the-recommended-dependencies


In [ ]:
#| hide

# def test_metrics_reloaded_binary():
#     # Native Backend Check
#     mr_bin_monai = MetricsReloadedBinary(metric_name="Accuracy", backend="monai")
#     test_eq(mr_bin_monai.__class__.__name__, "MetricsReloadedBinary")

#     # Fastai Wrapper Check
#     mr_bin_fastai = MetricsReloadedBinary(metric_name="Accuracy", backend="fastai")
#     test_eq(mr_bin_fastai.name, "MetricsReloadedBinary")
    
#     # Simulate a fastai iteration safely with (Batch, Channel, Spatial) -> (2, 1, 1)
#     pred = torchTensor([[[0.9]], [[0.1]]])
#     targ = torchTensor([[[1]], [[0]]])
#     learn = SimpleNamespace(pred=pred, y=targ, yb=(targ,), to_detach=lambda b: b)
    
#     mr_bin_fastai.reset()
#     mr_bin_fastai.accumulate(learn)
#     val = mr_bin_fastai.value
#     test_eq(val is not None, True)

#     return "MetricsReloadedBinary tests passed"

# test_eq(test_metrics_reloaded_binary(), "MetricsReloadedBinary tests passed")

#### `MetricsReloadedBinary` Parameter Reference

*A specialized wrapper for binary classification or segmentation tasks using the Metrics Reloaded framework.*

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`metric_name`** | `str \| list` | *Required* | The name or list of names of the metrics to compute (e.g., `"Accuracy"`, `"Dice"`). |
| **`backend`** | `str` | `"monai"` | *bioMONAI specific:* Target execution backend (`"monai"`, `"ignite"`, or `"fastai"`). |
| **`**kwargs`** | `dict` | `{}` | Additional arguments forwarded directly to the underlying `monai.metrics.MetricsReloadedBinary`. |

#### Example: Using MetricsReloadedCategorical in a fastai workflow

In [ ]:
# 1. Instantiate the Metric wrapper for multiclass (categorical) classification
mr_cat = MetricsReloadedCategorical(metric_name="Accuracy", backend="fastai")

# 2. Simulate categorical predictions and targets
# MONAI strictly expects at least 3 dimensions: (Batch, Channel, Spatial)
pred_cat = torchTensor([
    [[0.1], [0.9], [0.0]], # Predicts class 1 (Batch 0)
    [[0.8], [0.1], [0.1]], # Predicts class 0 (Batch 1)
    [[0.0], [0.2], [0.8]]  # Predicts class 2 (Batch 2)
]) # Shape: (3, 3, 1)

targ_cat = torchTensor([[[1]], [[0]], [[2]]]) # Shape: (3, 1, 1)

# 3. Mock the fastai learner object correctly for AvgMetric
mock_learn_cat = SimpleNamespace(
    pred=pred_cat, 
    y=targ_cat,
    yb=(targ_cat,),
    to_detach=lambda b: b
)

try:
    mr_cat.reset()
    mr_cat.accumulate(mock_learn_cat)

    print(f"Metrics Reloaded Categorical Name: {mr_cat.name}")
    print(f"Calculated Score: {mr_cat.value}")
except Exception as e:
    # Capturing exception in case the optional 'metricsreloaded' package is not installed
    print(f"Metrics Reloaded Exception caught: {e}")

Metrics Reloaded Exception caught: from MetricsReloaded.metrics.pairwise_measures import MultiClassPairwiseMeasures (No module named 'MetricsReloaded').

For details about installing the optional dependencies, please visit:
    https://docs.monai.io/en/latest/installation.html#installing-the-recommended-dependencies


In [ ]:
#| hide

def test_metrics_reloaded_categorical():
    # Native Backend Check
    mr_cat_monai = MetricsReloadedCategorical(metric_name="Accuracy", backend="monai")
    test_eq(mr_cat_monai.__class__.__name__, "MetricsReloadedCategorical")

    # Fastai Wrapper Check
    mr_cat_fastai = MetricsReloadedCategorical(metric_name="Accuracy", backend="fastai")
    test_eq(mr_cat_fastai.name, "MetricsReloadedCategorical")
    
    # Simulate a fastai iteration safely for categorical (B, C, Spatial)
    pred = torchTensor([
        [[0.2], [0.8]], 
        [[0.9], [0.1]]
    ]) # Shape (2, 2, 1)
    targ = torchTensor([[[1]], [[0]]]) # Shape (2, 1, 1)
    
    learn = SimpleNamespace(pred=pred, y=targ, yb=(targ,), to_detach=lambda b: b)
    
    try:
        mr_cat_fastai.reset()
        mr_cat_fastai.accumulate(learn)
        val = mr_cat_fastai.value
        test_eq(val is not None, True)
    except Exception:
        pass

    return "MetricsReloadedCategorical tests passed"

test_eq(test_metrics_reloaded_categorical(), "MetricsReloadedCategorical tests passed")

#### `MetricsReloadedCategorical` Parameter Reference

*A specialized wrapper for multiclass classification or segmentation tasks using the Metrics Reloaded framework.*

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`metric_name`** | `str \| list` | *Required* | The name or list of names of the metrics to compute. |
| **`backend`** | `str` | `"monai"` | *bioMONAI specific:* Target execution backend (`"monai"`, `"ignite"`, or `"fastai"`). |
| **`**kwargs`** | `dict` | `{}` | Additional arguments forwarded directly to the underlying `monai.metrics.MetricsReloadedCategorical`. |

## Fourier Ring Correlation

Fourier Ring Correlation (FRC) is a frequency-domain method used to quantify the similarity between two independent measurements of the same underlying signal. It is widely applied in fields such as microscopy, cryo-electron microscopy, and super-resolution imaging to estimate spatial resolution in a statistically robust manner. By comparing corresponding Fourier components over concentric rings (in 2D) or shells (in 3D) of equal spatial frequency, FRC provides a frequency-dependent correlation profile that reflects the reproducibility of structural information.

Mathematically, the Fourier ring correlation at spatial frequency ( r ) is defined as:

$$FRC(r) = \frac{\sum_{k \in r} F_1(k) \overline{F_2(k)}}{\sqrt{\left(\sum_{k \in r} |F_1(k)|^2\right) \left(\sum_{k \in r} |F_2(k)|^2\right)}}$$

where $ F_1(k) $ and $ F_2(k) $ are the Fourier transforms of the two independent images, $ k $ denotes frequency coordinates lying on a ring of radius $ r $, and the overline indicates complex conjugation. The numerator measures cross-correlation of corresponding Fourier coefficients, while the denominator normalizes by their respective spectral energies.

A function that calculates an FRC-based metric determines a resolution criterion by identifying where the FRC curve crosses the predefined threshold at 1/7.

The resulting FRC curve provides a frequency-resolved measure of signal consistency, and the derived cutoff frequency can be converted into a spatial resolution estimate.

#### `FRCMetric` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`image1`** | `torchTensor` | *Required* | First independent measurement or image tensor. |
| **`image2`** | `torchTensor` | *Required* | Second independent measurement or image tensor to compare against. |

In [ ]:
#| export

class FRCMetric(BioMetric):
    _default = partial(mm.LossMetric,loss_fn=FRCLoss)

In [ ]:
# Example: Calculating FRC-based metric between two image tensors

# 1. Simulate two independent 2D image tensors (Height, Width)
# Based on bioMONAI's internal implementation, FRC expects 2D spatial tensors directly.
image_a = torchrandn(32, 32)
image_b = torchrandn(32, 32)

# 2. Compute the FRC-derived metric
frc_metric = FRCMetric()
frc_score = frc_metric(image_a, image_b)
print(f"FRC Metric Score: {frc_score.item()}")

FRC Metric Score: 0.8226511478424072


In [ ]:
#| hide

def test_frc_metric_logic():
    img1 = torchones(32, 32)
    img2 = torchones(32, 32)
    
    score = FRCMetric()(img1, img2)
    test_eq(score is not None, True)
        
    return "FRCMetric logic tests passed"

test_eq(test_frc_metric_logic(), "FRCMetric logic tests passed")

In [ ]:
# Create a small pair of synthetic images
image1 = torchTensor([
    [0., 0., 0., 0., 0., 0., 0., 0.],
    [0., 0., 1., 1., 1., 1., 0., 0.],
    [0., 1., 2., 2., 2., 2., 1., 0.],
    [0., 1., 2., 3., 3., 2., 1., 0.],
    [0., 1., 2., 3., 3., 2., 1., 0.],
    [0., 1., 2., 2., 2., 2., 1., 0.],
    [0., 0., 1., 1., 1., 1., 0., 0.],
    [0., 0., 0., 0., 0., 0., 0., 0.],
])

image2 = torchTensor([
    [0., 0., 0., 0., 0., 0., 0., 0.],
    [0., 0., 1., 1., 1., 1., 0., 0.],
    [0., 1., 2., 2., 2., 2., 1., 0.],
    [0., 1., 2., 3., 4., 2., 1., 0.],
    [0., 1., 2., 4., 3., 2., 1., 0.],
    [0., 1., 2., 2., 2., 2., 1., 0.],
    [0., 0., 1., 1., 1., 1., 0., 0.],
    [0., 0., 0., 0., 0., 0., 0., 0.],
])

# Mock fastai learner
learn = TestLearner(
    preds=image1,
    targs=image2,
)

# Create the metric through BioMetric
metric = FRCMetric(backend="fastai")

# Reset and accumulate exactly as fastai does
metric.reset()
metric.accumulate(learn)

# The metric should produce a scalar value
value = metric.value

assert value is not None

print(f"FRC metric: {value}")

FRC metric: 0.09430021047592163


## Active Learning

In [ ]:
#| export
class VarianceMetric(BioMetric):
    _default = mm.VarianceMetric

In [ ]:
#| export
class LabelQualityScore(BioMetric):
    _default = mm.LabelQualityScore

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()